code by Huan Li

In [1]:
# paths.py
from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
two_COLUMN_DIR = PROJECT_ROOT / "data" / "speeches" / "pdf_batches" / "two_column"

## Define the start  and end of SG speech, and extract the content into txt files,and fitler non-speech content.

In [2]:
# find SG speeches
from langdetect import detect
import re
start_sg = re.compile(
    r"(?im)^\s*(?:\(?\s*)?(?:\d+\s*[\.\)]\s*)?(?:the\s+)?"
    r"(?:secreta(?:r|i|l|v)y(?:\s*[\-\u2010\u2011\u2012\u2013\u2014]\s*|\s+)general)"
    r"(?:\s*\([^)]*\))?\s*[:;]\s*"
)

next_speaker = re.compile(
    r"(?im)^\s*(?:\(?\s*)?(?:\d+\s*[\.\)]\s*)?"
    r"(?:"
        r"(?:"
            r"The\s+(?:President|Chairman|Chair|Rapporteur|Secretary(?:\s*[\-\u2010-\u2014]\s*General|(?:\s+General)?)|Representative)"
            r"|(?:Mr|Mrs|Ms|Miss|Mme|M|Sir|Dr|Ambassador|Minister|Prince|Princess|Lord)\.?"
        r")"
        r"(?:\s+[A-Z][\w'\-\.]{1,30},?){0,8}"
        r"(?:\s*\([^)]*\)){0,4}\s*[:;]\s*"
        r"|"
        r"The\s+meeting\s+(?:rose|was\s+adjourned|was\s+suspended|was\s+resumed)\b.*$"
    r")"
)


def is_english(text):
    try:
        return detect(text) == "en"
    except:
        return False


# keep only English lines in the SG speech body 
def clean_speech_body(body: str, is_english) -> str:
    """Only loop lines inside ONE extracted SG speech."""
    kept = []
    for line in body.splitlines():
        clean_line = line.strip()
        if not clean_line:
            continue
        if is_english(clean_line):
            kept.append(clean_line)
    return "\n".join(kept).strip()

# extract SG speeches in one pass
def extract_sg_speeches_one_pass(text: str, is_english):
    speeches = []
    for m in start_sg.finditer(text):
        start = m.end()
        stop = next_speaker.search(text, start)
        end = stop.start() if stop else len(text)

        header = m.group().strip()
        raw_body = text[start:end].strip()
        if not raw_body:
            continue

        cleaned_body = clean_speech_body(raw_body, is_english)
        if cleaned_body:  # drop speeches that become empty after filtering
            speeches.append((header, cleaned_body))
    return speeches



## Define the filenames of the txt files

In [3]:
batch_name = "batch_21"

# 0) paths
PROJECT_ROOT = Path.cwd()
in_path = PROJECT_ROOT / "data" / "speeches"\
      / "txt_from_image" / "two_columns" / f"two_column_{batch_name}.txt"

out_dir = PROJECT_ROOT / "data" / "speeches" / "extracted_from_image"
out_dir.mkdir(parents=True, exist_ok=True)

# prepare pdf names 
files = two_COLUMN_DIR / f"{batch_name}"
file_names_str = ""

for file in files.iterdir():
    if file.is_file():
        print(file.name)
        file_names_str += file.name + "\n"


S_1964_PV.1097_speeches.pdf
S_1964_PV.1102_speeches.pdf
S_1964_PV.1103_speeches.pdf
S_1964_PV.1121_speeches.pdf
S_1983_PV.2497_speeches.pdf
S_1984_PV.2519_speeches.pdf
S_1985_PV.2608_speeches.pdf


## Convert into one merged txt file, and check if the content is complete.


In [ ]:
# # --- usage ---

# text = in_path.read_text(encoding="utf-8", errors="replace")

# speeches = extract_sg_speeches_one_pass(text, is_english)
# print("Found SG speeches:", len(speeches))

# merged_path = out_dir / f"{in_path.stem}__SG_ALL.txt"
# merged_chunks = []
# for i, (header, body) in enumerate(speeches):
#     merged_chunks.append(
#         f"\n{'='*20}CHAPTER_{i:03d} {'='*20}\n"
#         f"{header}\n{body}\n"
#     )

# merged_path.write_text("".join(merged_chunks), encoding="utf-8")
# print("Saved merged file to:", merged_path)

Found SG speeches: 6
Saved merged file to: d:\PDS_UNSG_text_corpus\data\speeches\extracted_from_image\two_column_batch_21__SG_ALL.txt


## Given the filenames oftxt files, extract the content of SG speech, and save as separate txt files.

In [4]:
from collections import defaultdict

# 0) your pdf name list 
pdf_names_str = """
S_1964_PV.1097_speeches.pdf
S_1964_PV.1102_speeches.pdf
S_1964_PV.1103_speeches.pdf
S_1983_PV.2497_speeches.pdf
S_1984_PV.2519_speeches.pdf
S_1985_PV.2608_speeches.pdf
""".strip()

pdf_names = [line.strip() for line in pdf_names_str.splitlines() if line.strip()]


# parse pdf names
pdf_names = [line.strip() for line in pdf_names_str.splitlines() if line.strip()]

# read & extract
text = in_path.read_text(encoding="utf-8", errors="replace")
speeches = extract_sg_speeches_one_pass(text, is_english)

print("Found SG speeches:", len(speeches))
print("PDF names:", len(pdf_names))

if len(speeches) != len(pdf_names):
    raise ValueError(
        f"Mismatch: found {len(speeches)} speeches but {len(pdf_names)} pdf names."
    )

out_dir.mkdir(parents=True, exist_ok=True)

# dictionary to count occurrences
name_counter = defaultdict(int)

for i, (header, body) in enumerate(speeches):
    pdf = pdf_names[i]
    stem = Path(pdf).stem  # remove .pdf

    # increase counter
    name_counter[stem] += 1
    count = name_counter[stem]

    # first occurrence: no suffix
    if count == 1:
        filename = f"{stem}.txt"
    else:
        filename = f"{stem}_{count}.txt"

    out_path = out_dir / filename
    out_path.write_text(f"{header}\n{body}\n", encoding="utf-8")

print("Saved individual txt files to:", out_dir)


Found SG speeches: 6
PDF names: 6
Saved individual txt files to: d:\PDS_UNSG_text_corpus\data\speeches\extracted_from_image
